# 12. 비동기

요청을 여러 개 보낼 때, 하나씩 답을 기다리면 기다리는 시간이 쌓입니다. 기다리는 동안 다른 요청도 보내 두는 방법을 봅니다.

- 코드 칸 안의 `#>` 줄은 그 칸에서 할 일입니다
- 그 아래에 코드를 치고 Shift + Enter로 실행합니다

## 1. 하나씩 기다리기

09장에서 만든 시간 재기 데코레이터입니다.

In [ ]:
import time


def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print("걸린 시간:", end - start, "초")
        return result
    return wrapper

날씨 API(Open-Meteo)로 다섯 도시의 지금 기온을 받습니다. 키 없이 쓰는 공식 API이고, 답이 오는 데 시간이 조금 걸려 기다리는 시간이 잘 보입니다. 10장처럼 `requests.get`에 `params`를 넣습니다. 이번에는 값이 셋입니다.

| `params` 이름 | 넣을 값 |
| --- | --- |
| `"latitude"` | 위도 `lat` |
| `"longitude"` | 경도 `lon` |
| `"current"` | `"temperature_2m"` (지금 기온) |

지금 기온은 응답의 `["current"]["temperature_2m"]`에 있습니다. 도시 이름과 위도 · 경도는 주어진 칸에 있습니다.

In [ ]:
#> requests를 import하고, 위도 lat · 경도 lon을 받아 그곳의 지금 기온을 돌려주는 get_temp(lat, lon)을 만듭니다
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
CITIES = {"서울": (37.57, 126.98), "부산": (35.18, 129.08), "대구": (35.87, 128.60), "광주": (35.16, 126.85), "제주": (33.50, 126.53)}

In [ ]:
#> @timer를 붙인 get_all()이 CITIES의 도시마다 get_temp를 불러 {도시: 기온} 딕셔너리를 돌려주게 만들고, 불러 봅니다

요청을 보내고 → 답이 올 때까지 기다리고 → 다음 요청. 다섯 번 기다린 시간이 그대로 더해졌습니다. 기다리는 동안 프로그램은 아무것도 하지 않습니다.

## 2. `async def` · `await`

**기본 구조**

```
import asyncio

async def 함수이름(매개변수):
    결과 = await asyncio.to_thread(오래걸리는함수, 인수1, 인수2)
    return 결과

결과 = await 함수이름(인수)         # 노트북에서는 칸에서 바로 await
```

`asyncio.to_thread(함수, 인수…)`는 `requests.get`처럼 보통 함수를 `await`로 기다릴 수 있게 바꿔 부릅니다.

In [ ]:
#> asyncio를 import하고, get_temp를 asyncio.to_thread로 부르는 get_temp_async(lat, lon)을 만듭니다

In [ ]:
#> get_temp_async(37.57, 126.98)을 await 없이 불러 봅니다

`await` 없이 부르면 실행되지 않고 **코루틴**이라는 것만 돌아옵니다. 09장 제너레이터 함수를 부르면 실행 대신 제너레이터가 돌아온 것과 같습니다.

In [ ]:
#> 이번에는 await를 붙여 불러 봅니다

`await`를 붙이면 실행되고 결과가 나옵니다. 3절에서 코루틴 여러 개를 한꺼번에 기다립니다.

## 3. 한꺼번에: `asyncio.gather`

**기본 구조**

```
결과들 = await asyncio.gather(코루틴1, 코루틴2, 코루틴3)     # 결과들은 리스트, 넣은 순서대로
결과들 = await asyncio.gather(*코루틴_리스트)               # 리스트 앞에 *: 하나씩 풀어 넣기
```

`asyncio.gather`에 코루틴을 여러 개 넣으면 **한꺼번에** 시작하고, 모두 끝날 때까지 기다립니다.

In [ ]:
#> 다섯 도시의 get_temp_async를 asyncio.gather로 한꺼번에 기다리고(위도 · 경도는 1절 CITIES에서 복사), 앞뒤로 time.time()을 재서 결과와 걸린 시간을 출력합니다

1절보다 훨씬 빠릅니다. 다섯 요청이 **같이 기다렸기** 때문입니다. 걸린 시간은 가장 늦은 요청 하나쯤입니다.

도시가 많으면 하나씩 적기 번거롭습니다. 코루틴을 리스트로 만들고 앞에 `*`를 붙이면 하나씩 풀어서 넣습니다. 1절 `timer`의 `func(*args, **kwargs)`에 있던 `*`와 같은 모양입니다.

In [ ]:
#> CITIES의 위도 · 경도로 코루틴 리스트 jobs를 만들고, asyncio.gather(*jobs)로 기다려 도시 이름과 짝지어 출력합니다

4차시 FastAPI에서는 요청을 받는 함수를 `async def`로 씁니다. 서버가 한 사람의 요청을 기다리는 동안 다른 사람의 요청도 처리할 수 있게 하려는 것입니다.

## 정리

| 할 일 | 방법 |
| --- | --- |
| 부르면 코루틴이 돌아오는 함수 | `async def 함수(…):` |
| 기다리기 | `await 코루틴` (노트북에서는 칸에서 바로) |
| 보통 함수(`requests.get`)를 `await`로 기다리게 | `await asyncio.to_thread(함수, 인수…)` |
| 여러 개를 한꺼번에 | `await asyncio.gather(코루틴1, 코루틴2, …)` · 리스트면 `*리스트` |
| 걸린 시간 | `start = time.time()` … `time.time() - start` |

| 보이는 것 | 뜻 |
| --- | --- |
| `<coroutine object …>` | `await`를 빠뜨림 |

연습: `exercises/12_async.ipynb`